# Payment Analysis

This notebook analyzes customer payment behavior.

Focus areas:
- Payment method usage over time
- Number of orders by payment type
- Installment behavior
- Relationship between payment type and order volume

In [0]:
%sql
/*
Monthly Orders by Payment Type

Purpose:
    Analyze month-on-month order volume across different payment methods.
    This helps understand how customer payment preferences change over time.
*/

WITH monthly_payment_orders AS (
    SELECT
        DATE_TRUNC('MONTH', o.order_purchase_timestamp) AS order_month,
        p.payment_type,
        COUNT(DISTINCT o.order_id) AS order_volume
    FROM ecommerce_analysis.orders o
    LEFT JOIN ecommerce_analysis.payments p
        ON o.order_id = p.order_id
    WHERE o.order_purchase_timestamp IS NOT NULL
    GROUP BY
        DATE_TRUNC('MONTH', o.order_purchase_timestamp),
        p.payment_type
)

SELECT
    order_month,
    payment_type,
    order_volume,
    LAG(order_volume) OVER (
        PARTITION BY payment_type
        ORDER BY order_month
    ) AS previous_month_orders,
    order_volume - LAG(order_volume) OVER (
        PARTITION BY payment_type
        ORDER BY order_month
    ) AS month_over_month_change,
    ROUND(
        (order_volume - LAG(order_volume) OVER (
            PARTITION BY payment_type
            ORDER BY order_month
        )) * 100.0
        / LAG(order_volume) OVER (
            PARTITION BY payment_type
            ORDER BY order_month
        ),
        2
    ) AS month_over_month_percentage
FROM monthly_payment_orders
ORDER BY payment_type, order_month;

In [0]:
%sql
/*
Orders by Payment Installments

Purpose:
    Count orders based on the number of payment installments.
    This helps understand how often customers use installment-based payments.
*/

SELECT
    payment_installments,
    COUNT(DISTINCT order_id) AS order_volume,
    ROUND(AVG(payment_value), 2) AS avg_payment_value,
    ROUND(SUM(payment_value), 2) AS total_payment_value
FROM ecommerce_analysis.payments
GROUP BY payment_installments
ORDER BY payment_installments;

In [0]:
%sql
/*
Payment Type Summary

Purpose:
    Compare total orders and total payment value across payment methods.
*/

SELECT
    payment_type,
    COUNT(DISTINCT order_id) AS order_volume,
    ROUND(SUM(payment_value), 2) AS total_payment_value,
    ROUND(AVG(payment_value), 2) AS avg_payment_value
FROM ecommerce_analysis.payments
GROUP BY payment_type
ORDER BY total_payment_value DESC;